> **Versión corregida (v2.12, ejecución verificada)** — generada a partir de `02_entendimiento_datos_reinicio.ipynb (Drive/Colab Notebooks)` sin sobrescribirlo.
>
> Cambios aplicados:
> - Celda de inicialización sustituida por el bootstrap portable: las rutas fijas `/content/drive/MyDrive/PIDA-RL-Diabetes` ahora dependen de `PROJECT_ROOT`.
> - Renombrado semántico: el antiguo `PROJECT_ROOT` (copia del repositorio) pasa a `REPO_ROOT`; el antiguo `PERSISTENT_ROOT` (Drive) pasa a `PROJECT_ROOT` (se conserva `PERSISTENT_ROOT` como alias).
> - El clonado usa la rama `feature/entorno-capacidad-v2`; antes clonaba `main`, que no contiene `src/` V2.
>
> Rutas: `PROJECT_ROOT` = datos/modelos/resultados (Drive en Colab, o `PIDA_PROJECT_ROOT`); `REPO_ROOT` = código. Modo prueba: `PIDA_MODO_PRUEBA=1` (cohorte pequeña y pocos episodios).


# PIDA-RL Diabetes — 02. Entendimiento y calidad de datos

Este notebook inspecciona las fuentes ENSANUT cargadas en el notebook 01 para identificar variables candidatas y evaluar su calidad. **No crea la cohorte, no imputa valores y no modifica archivos fuente.**

El objetivo es producir evidencia auditable para decidir las variables que se utilizarán en `03_preparacion_cohorte.ipynb`.

## 1. Inicialización y recuperación de fuentes

Si ejecutas este notebook justo después del 01, reutilizará `frames` y `metas` presentes en memoria. Si se abrió una sesión nueva, recargará los archivos `.sav` localizados en la carpeta persistente `data/raw`.

In [ ]:
# === Bootstrap portable: PROJECT_ROOT (corrección v2.12) ===
# PROJECT_ROOT = raíz persistente de datos, modelos y resultados.
# REPO_ROOT    = copia local del código (repositorio GitHub).
# Prioridad: variables de entorno PIDA_PROJECT_ROOT / PIDA_REPO_ROOT,
# después Google Colab + Drive y, por último, ejecución local.
from pathlib import Path
from datetime import datetime, timezone
import os
import sys
import json
import importlib
import importlib.util
import subprocess

PROJECT_NAME = 'pida-rl-datos-publicos'

REPO_URL = 'https://github.com/JssSalas/pida-rl-datos-publicos.git'
BRANCH = os.environ.get('PIDA_BRANCH', 'feature/entorno-capacidad-v2')
PROJECT_FOLDER_NAME = 'PIDA-RL-Diabetes'
IN_COLAB = 'google.colab' in sys.modules

DRIVE_ROOT = None
if os.environ.get('PIDA_PROJECT_ROOT'):
    PROJECT_ROOT = Path(os.environ['PIDA_PROJECT_ROOT']).expanduser().resolve()
elif IN_COLAB:
    DRIVE_ROOT = next(
        (p for p in [Path('/content/drive'), Path('/content/gdrive')] if (p / 'MyDrive').is_dir()),
        None,
    )
    if DRIVE_ROOT is None:
        from google.colab import drive
        mount_point = Path('/content/drive')
        if mount_point.exists() and any(mount_point.iterdir()):
            mount_point = Path('/content/gdrive')
        drive.mount(str(mount_point), force_remount=False)
        DRIVE_ROOT = mount_point
    PROJECT_ROOT = DRIVE_ROOT / 'MyDrive' / PROJECT_FOLDER_NAME
else:
    PROJECT_ROOT = (Path.cwd() / PROJECT_FOLDER_NAME).resolve()
PROJECT_ROOT.mkdir(parents=True, exist_ok=True)


def _buscar_repo(inicio):
    for candidato in [inicio, *inicio.parents]:
        if (candidato / 'src' / 'environment' / 'diabetes_followup_env.py').is_file():
            return candidato
    return None


if os.environ.get('PIDA_REPO_ROOT'):
    REPO_ROOT = Path(os.environ['PIDA_REPO_ROOT']).expanduser().resolve()
else:
    REPO_ROOT = _buscar_repo(Path.cwd().resolve()) or (
        Path('/content/pida-rl-datos-publicos') if IN_COLAB
        else Path.cwd().resolve() / 'pida-rl-datos-publicos'
    )
if not REPO_ROOT.exists():
    subprocess.check_call(['git', 'clone', '--branch', BRANCH, '--single-branch', REPO_URL, str(REPO_ROOT)])
elif not (REPO_ROOT / 'src').is_dir():
    raise RuntimeError(f'REPO_ROOT no contiene la carpeta src: {REPO_ROOT}')
elif IN_COLAB and (REPO_ROOT / '.git').is_dir():
    # En Colab se alinea la copia temporal con la rama de trabajo (sin push ni commit).
    for args in (['fetch', 'origin', BRANCH], ['checkout', BRANCH], ['pull', '--ff-only', 'origin', BRANCH]):
        subprocess.run(['git', '-C', str(REPO_ROOT), *args], check=False)

SRC_ROOT = REPO_ROOT / 'src'
for ruta in (str(REPO_ROOT), str(SRC_ROOT)):
    if ruta not in sys.path:
        sys.path.insert(0, ruta)
os.chdir(REPO_ROOT)

# Estructura persistente común (antes: rutas fijas /content/drive/MyDrive/PIDA-RL-Diabetes).
PERSISTENT_ROOT = PROJECT_ROOT  # alias de compatibilidad con los notebooks originales
MYDRIVE_ROOT = PROJECT_ROOT.parent
DATA_ROOT = PROJECT_ROOT / 'data'
RAW_DATA_DIR = DATA_ROOT / 'raw'
INTERIM_DATA_DIR = DATA_ROOT / 'interim'
PROCESSED_DATA_DIR = DATA_ROOT / 'processed'
METADATA_DIR = DATA_ROOT / 'metadata'
SPECIFICATIONS_DIR = DATA_ROOT / 'specifications'
MODELS_DIR = PROJECT_ROOT / 'models'
RESULTS_DIR = PROJECT_ROOT / 'results'
COHORT_PATH = PROCESSED_DATA_DIR / 'cohorte_diabetes.csv'

# Modo prueba: cohorte pequeña y pocos episodios (PIDA_MODO_PRUEBA=1).
MODO_PRUEBA = os.environ.get('PIDA_MODO_PRUEBA', '0') == '1'
N_COHORTE_PRUEBA = int(os.environ.get('PIDA_N_COHORTE', '120'))
SEMILLA_MUESTRA_PRUEBA = 2026
RESULTS_V2_DIR = RESULTS_DIR / ('capacidad_v2_prueba' if MODO_PRUEBA else 'capacidad_v2')


def muestra_cohorte_prueba(cohorte, n=None, semilla=SEMILLA_MUESTRA_PRUEBA):
    """Devuelve (cohorte, factor). En modo prueba toma una submuestra
    estratificada por categoria_riesgo (bajo/medio/alto); no crea columnas."""
    n = N_COHORTE_PRUEBA if n is None else int(n)
    if not MODO_PRUEBA or len(cohorte) <= n:
        return cohorte, 1.0
    fraccion = n / len(cohorte)
    muestra = (
        cohorte.groupby('categoria_riesgo', group_keys=False)
        .sample(frac=fraccion, random_state=semilla)
        .sort_values('FOLIO_INT')
        .reset_index(drop=True)
    )
    return muestra, len(muestra) / len(cohorte)


def escalar_capacidad(capacidad, factor):
    """Mantiene la presión de capacidad (recursos/persona) al reducir la cohorte."""
    return capacidad if factor >= 1.0 else max(1, int(round(capacidad * factor)))

print('Entorno:', 'Google Colab' if IN_COLAB else 'local')
print('PROJECT_ROOT (datos/resultados):', PROJECT_ROOT)
print('REPO_ROOT (código):', REPO_ROOT)
print('Modo prueba:', MODO_PRUEBA)

for directory in [RAW_DATA_DIR, METADATA_DIR]:
    directory.mkdir(parents=True, exist_ok=True)
print('Datos crudos:', RAW_DATA_DIR)
print('Metadatos:', METADATA_DIR)


In [ ]:
import numpy as np
import pandas as pd

try:
    import pyreadstat
except ImportError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet', 'pyreadstat'])
    import pyreadstat

SOURCE_SPECS = {
    'adultos': {
        'expected_names': ['AdultosENSANUT2022.sav', 'Adultos_ENSANUT_2022.sav'],
        'keywords': ['adult', 'ensanut', '2022'],
        'required': True
    },
    'antro_tension': {
        'expected_names': ['AntroTensionENSANUT2022.sav', 'Antro_Tension_ENSANUT_2022.sav'],
        'keywords': ['antro', 'tension'],
        'required': False
    },
    'sangre': {
        'expected_names': ['MuestrasSangreENSANUT2022.sav', 'Muestras_Sangre_ENSANUT_2022.sav'],
        'keywords': ['sangre'],
        'required': False
    }
}

def locate_source(spec, search_dir):
    all_files = [path for path in search_dir.rglob('*.sav') if path.is_file()]
    for expected_name in spec['expected_names']:
        exact = [path for path in all_files if path.name.lower() == expected_name.lower()]
        if exact:
            return exact[0]
    keywords = [item.lower() for item in spec['keywords']]
    matches = [
        path for path in all_files
        if all(keyword in path.name.lower() for keyword in keywords)
    ]
    return matches[0] if len(matches) == 1 else None

if 'frames' not in globals() or 'metas' not in globals() or 'adultos' not in frames:
    print('No se encontraron fuentes en memoria; se recargarán desde data/raw.')
    frames = {}
    metas = {}
    for name, spec in SOURCE_SPECS.items():
        path = locate_source(spec, RAW_DATA_DIR)
        if path is None:
            if spec['required']:
                raise FileNotFoundError(
                    f'No se encontró la fuente obligatoria {name!r} en {RAW_DATA_DIR}. '
                    'Ejecuta 01_fuentes_y_carga_reinicio.ipynb o revisa la ubicación del archivo.'
                )
            print(f'{name}: fuente opcional no localizada.')
            continue
        frames[name], metas[name] = pyreadstat.read_sav(path, apply_value_formats=False)
        print(f'{name}: {frames[name].shape}')
else:
    print('Se reutilizan frames y metas de la sesión actual.')

df_adultos = frames['adultos']
df_diabetes = df_adultos

print('Adultos:', df_adultos.shape)
print('Fuentes disponibles:', list(frames.keys()))


## 2. Inventario estructural de fuentes

Este inventario identifica dimensiones, nombres de variables, tipos y claves de unión sin interpretar todavía los códigos de las variables.

In [ ]:
source_summary = []

for source_name, df in frames.items():
    candidate_keys = [
        column for column in df.columns
        if column.upper() in {'FOLIO_I', 'FOLIO_INT'}
    ]
    source_summary.append({
        'fuente': source_name,
        'filas': int(df.shape[0]),
        'columnas': int(df.shape[1]),
        'claves_detectadas': ', '.join(candidate_keys),
        'duplicados_folio_int': (
            int(df['FOLIO_INT'].duplicated().sum())
            if 'FOLIO_INT' in df.columns else None
        )
    })

df_source_summary = pd.DataFrame(source_summary)
display(df_source_summary)

print('Primeras 60 variables de Adultos:')
print(df_adultos.columns.tolist()[:60])


## 3. Búsqueda auditable de variables candidatas

Los patrones son solo un mecanismo de descubrimiento. Una variable se incorporará a la cohorte únicamente después de revisar su etiqueta, codificación y significado en el cuestionario/diccionario oficial.

In [ ]:
SEARCH_PATTERNS = {
    'identificador': ['folio'],
    'edad': ['edad', 'ed_'],
    'sexo': ['sexo', 'sex_'],
    'diabetes': ['diab', 'diabet', 'gluco', 'insulin'],
    'hipertension': ['hiper', 'tension', 'presion'],
    'complicaciones': ['complic', 'renal', 'retino', 'neurop', 'amputa'],
    'comorbilidades': ['cardio', 'infarto', 'cerebro', 'obes', 'cancer', 'depre'],
    'hospitalizacion': ['hospital', 'urgenc', 'intern'],
    'atencion_salud': ['consulta', 'medic', 'trat', 'control', 'servicio'],
    'antropometria': ['peso', 'talla', 'imc', 'cintura'],
    'laboratorio': ['gluco', 'hba1c', 'hemoglob', 'colesterol', 'triglic']
}

def metadata_label(meta, variable):
    labels = getattr(meta, 'column_labels', None)
    names = getattr(meta, 'column_names', None)
    if labels is not None and names is not None and variable in names:
        return labels[names.index(variable)]
    return None

candidate_rows = []
for source_name, df in frames.items():
    meta = metas.get(source_name)
    for variable in df.columns:
        variable_text = str(variable).lower()
        label = metadata_label(meta, variable) if meta is not None else None
        label_text = str(label).lower() if label else ''
        matched_groups = [
            group for group, patterns in SEARCH_PATTERNS.items()
            if any(pattern in variable_text or pattern in label_text for pattern in patterns)
        ]
        if matched_groups:
            series = df[variable]
            candidate_rows.append({
                'fuente': source_name,
                'variable': variable,
                'etiqueta': label,
                'grupos_candidatos': '; '.join(matched_groups),
                'tipo': str(series.dtype),
                'n_registros': int(series.shape[0]),
                'n_faltantes': int(series.isna().sum()),
                'porcentaje_faltantes': round(float(series.isna().mean() * 100), 2),
                'n_unicos': int(series.nunique(dropna=True))
            })

df_candidates = pd.DataFrame(candidate_rows)
df_candidates = df_candidates.sort_values(['fuente', 'grupos_candidatos', 'variable']).reset_index(drop=True)

print('Variables candidatas encontradas:', len(df_candidates))
display(df_candidates.head(100))


## 4. Perfil de calidad de variables candidatas

Se inspeccionan valores frecuentes, intervalos numéricos y posibles códigos especiales. Los valores extremos o códigos como 9, 99, 999, etc. se reportan, pero **no se recodifican en este notebook**.

In [ ]:
def profile_variable(df, variable, top_n=10):
    series = df[variable]
    profile = {
        'n': int(len(series)),
        'faltantes': int(series.isna().sum()),
        'porcentaje_faltantes': round(float(series.isna().mean() * 100), 2),
        'n_unicos': int(series.nunique(dropna=True)),
        'tipo': str(series.dtype)
    }

    if pd.api.types.is_numeric_dtype(series):
        finite = series.replace([np.inf, -np.inf], np.nan).dropna()
        profile.update({
            'n_inf': int(np.isinf(series.dropna()).sum()),
            'minimo': float(finite.min()) if not finite.empty else None,
            'p01': float(finite.quantile(0.01)) if not finite.empty else None,
            'mediana': float(finite.median()) if not finite.empty else None,
            'p99': float(finite.quantile(0.99)) if not finite.empty else None,
            'maximo': float(finite.max()) if not finite.empty else None
        })

    frequent = (
        series.value_counts(dropna=False)
        .head(top_n)
        .rename_axis('valor')
        .reset_index(name='frecuencia')
    )
    return profile, frequent

variables_prioritarias = (
    df_candidates[
        df_candidates['fuente'].eq('adultos')
        & df_candidates['grupos_candidatos'].str.contains(
            'edad|sexo|diabetes|complicaciones|comorbilidades|hospitalizacion|atencion_salud',
            regex=True
        )
    ]['variable']
    .drop_duplicates()
    .tolist()
)

print('Variables prioritarias a perfilar:', len(variables_prioritarias))

quality_rows = []
frequent_values = {}

for variable in variables_prioritarias:
    profile, frequent = profile_variable(df_adultos, variable)
    quality_rows.append({'fuente': 'adultos', 'variable': variable, **profile})
    frequent_values[variable] = frequent.to_dict(orient='records')

df_quality = pd.DataFrame(quality_rows)
display(df_quality.sort_values(['porcentaje_faltantes', 'variable']).head(100))


## 5. Revisar etiquetas y valores frecuentes

Usa esta celda para revisar las variables candidatas antes de seleccionar la especificación de cohorte. Cambia `VARIABLES_A_REVISAR` por nombres reales tomados de la tabla previa.

In [ ]:
VARIABLES_A_REVISAR = variables_prioritarias[:20]

for variable in VARIABLES_A_REVISAR:
    label = metadata_label(metas.get('adultos'), variable)
    profile, frequent = profile_variable(df_adultos, variable, top_n=15)
    print('\n' + '=' * 90)
    print('Variable:', variable)
    print('Etiqueta:', label)
    print('Perfil:', profile)
    display(frequent)


## 6. Exportar evidencia para la preparación de cohorte

Los archivos exportados no contienen microdatos. Sirven para documentar la selección posterior de variables y las decisiones de recodificación.

In [ ]:
candidates_path = METADATA_DIR / 'diccionario_variables_candidatas.csv'
quality_path = METADATA_DIR / 'perfil_calidad_variables_candidatas.csv'
frequent_path = METADATA_DIR / 'valores_frecuentes_variables_candidatas.json'
sources_path = METADATA_DIR / 'resumen_fuentes.csv'

df_candidates.to_csv(candidates_path, index=False, encoding='utf-8-sig')
df_quality.to_csv(quality_path, index=False, encoding='utf-8-sig')
df_source_summary.to_csv(sources_path, index=False, encoding='utf-8-sig')

with frequent_path.open('w', encoding='utf-8') as file:
    json.dump(frequent_values, file, ensure_ascii=False, indent=2, default=str)

print('Archivos generados:')
for path in [candidates_path, quality_path, frequent_path, sources_path]:
    print(' -', path, '| existe:', path.is_file())


## 7. Decisión requerida antes del notebook 03

Antes de construir la cohorte, documenta una especificación explícita con los nombres reales de las variables seleccionadas, la codificación de respuestas válidas y el tratamiento de faltantes/códigos especiales.

En la siguiente etapa no se deben usar variables solo por coincidencia de nombre. La selección deberá basarse en el diccionario oficial de ENSANUT y en el perfil de calidad exportado.